<a href="https://colab.research.google.com/github/tishamondal-stargazer/Machine-Learning/blob/main/Practical-07-Support-Vector-Machine/Practical-07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CS23059 Tisha Mondal

ML Lab-7

AIM: To apply Support Vector Machine for classification and analyze the impact of different kernels on model accuracy.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
url = "https://raw.githubusercontent.com/tishamondal-stargazer/Machine-Learning/refs/heads/main/datasets/pulsar_stars.csv"
df = pd.read_csv(url)
df.head(5)

,Mean of the integrated profile,Standard deviation of the integrated profile,Excess kurtosis of the integrated profile,Skewness of the integrated profile,Mean of the DM-SNR curve,Standard deviation of the DM-SNR curve,Excess kurtosis of the DM-SNR curve,Skewness of the DM-SNR curve,target_class
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225,0
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580,0
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909,0
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661,0
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306,0


In [4]:
df.shape

(17898, 9)

In [5]:
df.columns

Index([' Mean of the integrated profile',
       ' Standard deviation of the integrated profile',
       ' Excess kurtosis of the integrated profile',
       ' Skewness of the integrated profile', ' Mean of the DM-SNR curve',
       ' Standard deviation of the DM-SNR curve',
       ' Excess kurtosis of the DM-SNR curve', ' Skewness of the DM-SNR curve',
       'target_class'],
      dtype='object')

In [6]:
new_columns = {
    'Mean of the integrated profile': 'IP_Mean',
    'Standard deviation of the integrated profile': 'IP_Std',
    'Excess kurtosis of the integrated profile': 'IP_Kurtosis',
    'Skewness of the integrated profile': 'IP_Skewness',
    'Mean of the DM-SNR curve': 'DM_Mean',
    'Standard deviation of the DM-SNR curve': 'DM_Std',
    'Excess kurtosis of the DM-SNR curve': 'DM_Kurtosis',
    'Skewness of the DM-SNR curve': 'DM_Skewness',
    'target_class': 'Class'
}
df.columns = df.columns.str.strip()
df.columns = df.columns.str.strip()

# Then, rename using the provided mapping.
# The keys in new_columns should now match the stripped column names.
df.rename(columns=new_columns, inplace=True)


print(df.columns)

df.describe()

Index(['IP_Mean', 'IP_Std', 'IP_Kurtosis', 'IP_Skewness', 'DM_Mean', 'DM_Std',
       'DM_Kurtosis', 'DM_Skewness', 'Class'],
      dtype='object')


,IP_Mean,IP_Std,IP_Kurtosis,IP_Skewness,DM_Mean,DM_Std,DM_Kurtosis,DM_Skewness,Class
count,17898.000000,17898.000000,17898.000000,17898.000000,17898.000000,17898.000000,17898.000000,17898.000000,17898.000000
mean,111.079968,46.549532,0.477857,1.770279,12.614400,26.326515,8.303556,104.857709,0.091574
std,25.652935,6.843189,1.064040,6.167913,29.472897,19.470572,4.506092,106.514540,0.288432
min,5.812500,24.772042,-1.876011,-1.791886,0.213211,7.370432,-3.139270,-1.976976,0.000000
25%,100.929688,42.376018,0.027098,-0.188572,1.923077,14.437332,5.781506,34.960504,0.000000
50%,115.078125,46.947479,0.223240,0.198710,2.801839,18.461316,8.433515,83.064556,0.000000
75%,127.085938,51.023202,0.473325,0.927783,5.464256,28.428104,10.702959,139.309330,0.000000
max,192.617188,98.778911,8.069522,68.101622,223.392141,110.642211,34.539844,1191.000837,1.000000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17898 entries, 0 to 17897
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   IP_Mean      17898 non-null  float64
 1   IP_Std       17898 non-null  float64
 2   IP_Kurtosis  17898 non-null  float64
 3   IP_Skewness  17898 non-null  float64
 4   DM_Mean      17898 non-null  float64
 5   DM_Std       17898 non-null  float64
 6   DM_Kurtosis  17898 non-null  float64
 7   DM_Skewness  17898 non-null  float64
 8   Class        17898 non-null  int64  
dtypes: float64(8), int64(1)
memory usage: 1.2 MB


In [8]:
df.isnull().any().any()

np.False_

In [9]:
df[df.isnull().any(axis = 1)]

,IP_Mean,IP_Std,IP_Kurtosis,IP_Skewness,DM_Mean,DM_Std,DM_Kurtosis,DM_Skewness,Class


In [10]:
df.isnull().sum()

,0
IP_Mean,0
IP_Std,0
IP_Kurtosis,0
IP_Skewness,0
DM_Mean,0
DM_Std,0
DM_Kurtosis,0
DM_Skewness,0
Class,0


In [11]:
X=df.drop(columns=['Class'])
y=df['Class']

In [12]:
X.head()

,IP_Mean,IP_Std,IP_Kurtosis,IP_Skewness,DM_Mean,DM_Std,DM_Kurtosis,DM_Skewness
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306


In [13]:
X.columns

Index(['IP_Mean', 'IP_Std', 'IP_Kurtosis', 'IP_Skewness', 'DM_Mean', 'DM_Std',
       'DM_Kurtosis', 'DM_Skewness'],
      dtype='object')

In [14]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.7,test_size=0.3,random_state=42)

In [15]:
X_train.shape,X_test.shape

((12528, 8), (5370, 8))

In [16]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [17]:
X_train

array([[ 0.39398903,  0.27225212, -0.09759786, ..., -0.40338108,
        -0.19481754, -0.37780071],
       [-1.32725743, -1.52127273,  0.21878252, ..., -0.44578706,
         0.23837256, -0.02857034],
       [ 0.76130595,  0.98636783, -0.3245843 , ..., -0.2004973 ,
         0.038556  , -0.29531727],
       ...,
       [ 0.19435365, -0.48635278,  0.17242235, ..., -0.75001383,
         1.92294752,  1.91898259],
       [ 0.95911539,  0.50175114, -0.53401036, ..., -0.24457614,
        -0.20616758, -0.4040674 ],
       [ 0.37725131,  0.58093396, -0.12507761, ..., -0.74159513,
         2.05677923,  2.08022728]])

In [18]:
X_test

array([[ 0.22843774,  0.35277695, -0.27522748, ..., -0.43728381,
        -0.04649179, -0.25085816],
       [-1.38112247, -1.78437423,  1.44956147, ..., -0.22432742,
        -0.28338031, -0.46408179],
       [-0.30260453,  0.0698727 ,  0.02318325, ..., -0.54553131,
         0.21981851, -0.00614623],
       ...,
       [ 0.97889633,  0.3533328 , -0.46793554, ..., -0.3882352 ,
        -0.10691249, -0.33847727],
       [-0.17691945,  0.37417406, -0.21925266, ..., -0.55602557,
         0.1487237 , -0.05046816],
       [-0.01410706, -1.21585526, -0.01840983, ..., -0.20872688,
        -0.52776711, -0.61373985]])

In [19]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score  # Missing import

# 1. Initialize the model
svc = SVC()

# 2. Train the model
svc.fit(X_train, y_train)

# 3. Make predictions
y_pred = svc.predict(X_test)

# 4. Print accuracy
print('Model accuracy score with default hyperparameters: {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with default hyperparameters: 0.9790


In [20]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score  # Missing import

# 1. Initialize the model
svc = SVC(C=100.0)

# 2. Train the model
svc.fit(X_train, y_train)

# 3. Make predictions
y_pred = svc.predict(X_test)

# 4. Print accuracy
print('Model accuracy score with default hyperparameters: {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with default hyperparameters: 0.9803


In [21]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score  # Missing import

# 1. Initialize the model
svc = SVC(C=1000.0)

# 2. Train the model
svc.fit(X_train, y_train)

# 3. Make predictions
y_pred = svc.predict(X_test)

# 4. Print accuracy
print('Model accuracy score with default hyperparameters: {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with default hyperparameters: 0.9797


In [22]:
linear = SVC(kernel='linear')
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)
print('Model accuracy score with linear kernel and C=1.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with linear kernel and C=1.0 : 0.9788


In [23]:
linear = SVC(kernel='linear',C=100.0)
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)
print('Model accuracy score with linear kernel and C=1.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with linear kernel and C=1.0 : 0.9786


In [24]:
linear = SVC(kernel='linear',C=1000.0)
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)
print('Model accuracy score with linear kernel and C=1.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with linear kernel and C=1.0 : 0.9786


In [25]:
poly = SVC(kernel='poly')
poly.fit(X_train, y_train)
y_pred = poly.predict(X_test)
print('Model accuracy score with polynomial kernel and C=1.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with polynomial kernel and C=1.0 : 0.9769


In [26]:
# Change C to 100.0 (or any value like 0.1, 10.0, etc.)
poly = SVC(kernel='poly', C=100.0)

poly.fit(X_train, y_train)
y_pred = poly.predict(X_test)

print('Model accuracy score with polynomial kernel and C=100.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred)))


Model accuracy score with polynomial kernel and C=100.0 : 0.9793


In [27]:
# Change C to 100.0 (or any value like 0.1, 10.0, etc.)
poly = SVC(kernel='poly', C=1000.0)

poly.fit(X_train, y_train)
y_pred = poly.predict(X_test)

print('Model accuracy score with polynomial kernel and C=100.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred)))

Model accuracy score with polynomial kernel and C=100.0 : 0.9793


In [28]:
sigmoid = SVC(kernel='sigmoid',C=1.0)
sigmoid.fit(X_train, y_train)

SVC(kernel='sigmoid')

In [29]:
sigmoid = SVC(kernel='sigmoid',C=100.0)
sigmoid.fit(X_train, y_train)

SVC(C=100.0, kernel='sigmoid')

In [30]:
sigmoid = SVC(kernel='sigmoid',C=1000.0)
sigmoid.fit(X_train, y_train)

SVC(C=1000.0, kernel='sigmoid')

In [31]:
# Make prediction using sigmoid kernel (C=1000.0)
y_pred_sigmoid = sigmoid.predict(X_test)

print('Model accuracy score with sigmoid kernel and C=1000.0 : {0:0.4f}'.format(accuracy_score(y_test, y_pred_sigmoid)))

Model accuracy score with sigmoid kernel and C=1000.0 : 0.8773


In [32]:
#Accuracy Score Classification report and confusion matrix
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy_score(y_test, y_pred)

0.9793296089385475

In [33]:
print('Confusion Matrix:')
cm=confusion_matrix(y_test,y_pred)
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[4862   22]
 [  89  397]]


In [34]:
print(cm)

[[4862   22]
 [  89  397]]


In [35]:
print('classification report')
print(classification_report(y_test, y_pred))

classification report
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      4884
           1       0.95      0.82      0.88       486

    accuracy                           0.98      5370
   macro avg       0.96      0.91      0.93      5370
weighted avg       0.98      0.98      0.98      5370

